In [1]:
from pyspark.sql import SparkSession
import os

BASE_DIR = "/app"
DATA_RAW = os.path.join(BASE_DIR, "data", "raw")
DATA_INTERIM = os.path.join(BASE_DIR, "data", "interim")

spark = (
    SparkSession.builder
    .master("local[*]")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .appName("DLD_Valuation_Cleaning")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark ready | version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/12 19:11:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark ready | version: 4.1.1


In [2]:
csv_path = os.path.join(DATA_RAW, "Valuation.csv")

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("nullValue", "null")   # treats literal "null" strings as real nulls
    .csv(csv_path)
)

In [3]:
df.show(5)

+------------+-----------------+-------------------+--------------+----------------+-------------+------------+---------------+--------------+----------------+----------------+----------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+-----------+--------------------+
|procedure_id|procedure_name_ar|  procedure_name_en|procedure_year|procedure_number|instance_date|actual_worth|row_status_code|procedure_area|property_type_id|property_type_ar|property_type_en|property_sub_type_id|property_sub_type_ar|property_sub_type_en|area_id|        area_name_ar|        area_name_en|actual_area|property_total_value|
+------------+-----------------+-------------------+--------------+----------------+-------------+------------+---------------+--------------+----------------+----------------+----------------+--------------------+--------------------+--------------------+-------+--------------------+--------------------+-----------+--

In [4]:
print(len(df.columns))
print(df.columns)
df.printSchema()

20
['procedure_id', 'procedure_name_ar', 'procedure_name_en', 'procedure_year', 'procedure_number', 'instance_date', 'actual_worth', 'row_status_code', 'procedure_area', 'property_type_id', 'property_type_ar', 'property_type_en', 'property_sub_type_id', 'property_sub_type_ar', 'property_sub_type_en', 'area_id', 'area_name_ar', 'area_name_en', 'actual_area', 'property_total_value']
root
 |-- procedure_id: string (nullable = true)
 |-- procedure_name_ar: string (nullable = true)
 |-- procedure_name_en: string (nullable = true)
 |-- procedure_year: string (nullable = true)
 |-- procedure_number: string (nullable = true)
 |-- instance_date: string (nullable = true)
 |-- actual_worth: string (nullable = true)
 |-- row_status_code: string (nullable = true)
 |-- procedure_area: string (nullable = true)
 |-- property_type_id: string (nullable = true)
 |-- property_type_ar: string (nullable = true)
 |-- property_type_en: string (nullable = true)
 |-- property_sub_type_id: string (nullable = tru

In [5]:
from pyspark.sql.types import DoubleType, IntegerType

double_cols = ["property_total_value", "actual_area", "actual_worth", "procedure_area"]
int_cols = ["procedure_id", "procedure_year", "procedure_number"]

for c in double_cols:
    df = df.withColumn(c, df[c].cast(DoubleType()))

for c in int_cols:
    df = df.withColumn(c, df[c].cast(IntegerType()))

#changes ids from strings to integers as we need them to be int to not deal with formatting issues
id_cols = ["property_type_id", "property_sub_type_id", "area_id"]
for c in id_cols:
    df = df.withColumn(c, df[c].cast(IntegerType()))

In [6]:
from pyspark.sql.functions import to_timestamp

df = df.withColumn(
    "instance_date",
    to_timestamp("instance_date", "dd-MM-yyyy")
)

In [7]:
from pyspark.sql.functions import col, count, when

df.printSchema()
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show(truncate=False)

root
 |-- procedure_id: integer (nullable = true)
 |-- procedure_name_ar: string (nullable = true)
 |-- procedure_name_en: string (nullable = true)
 |-- procedure_year: integer (nullable = true)
 |-- procedure_number: integer (nullable = true)
 |-- instance_date: timestamp (nullable = true)
 |-- actual_worth: double (nullable = true)
 |-- row_status_code: string (nullable = true)
 |-- procedure_area: double (nullable = true)
 |-- property_type_id: integer (nullable = true)
 |-- property_type_ar: string (nullable = true)
 |-- property_type_en: string (nullable = true)
 |-- property_sub_type_id: integer (nullable = true)
 |-- property_sub_type_ar: string (nullable = true)
 |-- property_sub_type_en: string (nullable = true)
 |-- area_id: integer (nullable = true)
 |-- area_name_ar: string (nullable = true)
 |-- area_name_en: string (nullable = true)
 |-- actual_area: double (nullable = true)
 |-- property_total_value: double (nullable = true)



[Stage 2:>                                                          (0 + 5) / 5]

+------------+-----------------+-----------------+--------------+----------------+-------------+------------+---------------+--------------+----------------+----------------+----------------+--------------------+--------------------+--------------------+-------+------------+------------+-----------+--------------------+
|procedure_id|procedure_name_ar|procedure_name_en|procedure_year|procedure_number|instance_date|actual_worth|row_status_code|procedure_area|property_type_id|property_type_ar|property_type_en|property_sub_type_id|property_sub_type_ar|property_sub_type_en|area_id|area_name_ar|area_name_en|actual_area|property_total_value|
+------------+-----------------+-----------------+--------------+----------------+-------------+------------+---------------+--------------+----------------+----------------+----------------+--------------------+--------------------+--------------------+-------+------------+------------+-----------+--------------------+
|0           |0                |0 

In [8]:
df = df.fillna({
    "actual_area": 0,
    "actual_worth": 0,
    "property_total_value": 0
})

df = df.fillna("Unknown", subset=["property_sub_type_en", "area_name_en"])

#This will flag any missing values
df = df.withColumn("actual_area_missing", col("actual_area") == 0)

In [9]:
columns_to_drop = [
    "procedure_name_ar",
    "area_name_ar",
    "property_type_ar",
    "property_sub_type_ar",
    "row_status_code"
]

df = df.drop(*columns_to_drop)

In [10]:
df = df.dropDuplicates()

In [11]:
df = df.drop("procedure_id")

df = df.fillna({"procedure_area": 0.0})

df = df.fillna({"property_sub_type_id": 0})

In [12]:
df.show(10, truncate=False)
df.printSchema()
print("Final row count:", df.count())

+-------------------+--------------+----------------+-------------------+------------+--------------+----------------+----------------+--------------------+--------------------+-------+------------------+-----------+--------------------+-------------------+
|procedure_name_en  |procedure_year|procedure_number|instance_date      |actual_worth|procedure_area|property_type_id|property_type_en|property_sub_type_id|property_sub_type_en|area_id|area_name_en      |actual_area|property_total_value|actual_area_missing|
+-------------------+--------------+----------------+-------------------+------------+--------------+----------------+----------------+--------------------+--------------------+-------+------------------+-----------+--------------------+-------------------+
|Property Evaluation|2004          |354             |2004-05-16 00:00:00|1200000.0   |211.35        |1               |Land            |62                  |Commercial          |233    |Hor Al Anz        |211.34     |1200000.0 

In [13]:
import glob
import shutil

out_dir = os.path.join(DATA_INTERIM, "valuation_cleaned_updated_spark")
df.coalesce(1).write.mode("overwrite").option("header", "true").csv(out_dir)

part_file = glob.glob(os.path.join(out_dir, "part-*.csv"))[0]
final_csv = os.path.join(DATA_INTERIM, "valuation_cleaned_updated.csv")
shutil.copy(part_file, final_csv)

print("Wrote:", final_csv)


Wrote: /app/data/interim/valuation_cleaned_updated.csv
